In [1]:
# 任务0：环境配置
import pandas as pd
import numpy as np
import jieba
from gensim.models import Word2Vec
import warnings
warnings.filterwarnings('ignore')

print("环境配置完成")

D:\an\envs\nlp_zjw\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


环境配置完成


In [2]:
# 任务前置：数据加载与中文分词
# 加载餐厅评价数据集（列名comment对应train.csv表头）
df = pd.read_csv("train.csv")
print("数据预览（前5条）：")
print(df.head())

# 对每条评论分词，生成Word2Vec训练语料
sentences = []
for text in df['comment']:
    words = jieba.lcut(str(text))
    sentences.append(words)

print(f"\n分词完成，共{len(sentences)}条句子，示例：")
print(sentences[0])

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\钟浚伟\AppData\Local\Temp\jieba.cache


数据预览（前5条）：
   label                                            comment
0      0                                 一如既往地好吃，希望可以开到其他城市
1      0                                  味道很不错，分量足，客人很多，满意
2      0  下雨天来的，没有想象中那么火爆。环境非常干净，古色古香的，我自己也是个做服务行业的，我都觉得...
3      0                                    真心不好吃 基本上没得好多味道
4      0              少送一个牛肉汉堡 而且也不好吃 特别是鸡肉卷 **都不想评论了 谁买谁知道


Loading model cost 0.852 seconds.
Prefix dict has been built successfully.



分词完成，共10000条句子，示例：
['一如既往', '地', '好吃', '，', '希望', '可以', '开', '到', '其他', '城市']


In [3]:
# 任务1：使用Skip-Gram训练Word2Vec模型
# sg=1 代表Skip-Gram模式（任务要求，区别于默认CBOW）
model = Word2Vec(
    sentences=sentences,
    sg=1,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    epochs=10
)
model.save("word2vec_skipgram.model")
print("Skip-Gram模型训练完成！词汇表大小：", len(model.wv))

Skip-Gram模型训练完成！词汇表大小： 11780


In [4]:
# 任务2：输出“环境”的词向量及其形状
word = "环境"
vec = model.wv[word]

print(f"「{word}」的词向量：\n{vec}")
print(f"\n「{word}」的词向量形状：{vec.shape}")

「环境」的词向量：
[-1.96342394e-01 -9.89424996e-03 -9.91515964e-02  8.89602583e-03
 -8.33859667e-02 -9.25446749e-01  1.29636765e-01  5.66780865e-01
 -1.13936149e-01 -1.35211602e-01  7.64058232e-02 -4.42866296e-01
 -9.17005017e-02  2.85039544e-01 -1.16463244e-01  2.47066095e-02
  1.57307193e-01 -6.45140372e-03  1.82361111e-01 -1.54476285e+00
 -1.78361268e-04  5.85291721e-02 -2.53670663e-01 -4.36091065e-01
 -3.38377841e-02 -4.47040983e-02 -8.45116019e-01  3.72814417e-01
 -1.59665748e-01 -2.28946090e-01  4.35712636e-01  5.44285595e-01
  1.69008970e-01 -6.66289866e-01 -1.34202138e-01  4.41300571e-01
  4.60880905e-01 -1.64507300e-01  3.86362642e-01 -6.12802744e-01
  1.32642984e-01 -2.76667237e-01  2.45685820e-02 -3.84501904e-01
  1.33579567e-01 -7.23357975e-01 -2.44150907e-01  4.58663814e-02
  8.05902898e-01 -1.63511500e-01  1.05755314e-01  2.36402377e-02
  2.62457222e-01  9.66729403e-01  3.61310214e-01  1.39776736e-01
 -3.83775324e-01 -3.57851088e-01 -2.89220333e-01  2.74235487e-01
 -2.02705916e-0

In [5]:
# 任务3：输出与“好吃”语义最接近的3个词
similar_words = model.wv.most_similar("好吃", topn=3)
print("与「好吃」语义最接近的3个词：")
for i, (w, sim) in enumerate(similar_words, 1):
    print(f"{i}. {w}，相似度：{sim:.4f}")

与「好吃」语义最接近的3个词：
1. 名不虚传，相似度：0.7554
2. 很香，相似度：0.7546
3. 油腻，相似度：0.7446


In [6]:
# 任务4：计算两组词的相似度
# 计算「好吃」和「美味」的相似度
sim1 = model.wv.similarity("好吃", "美味")
# 计算「好吃」和「蟑螂」的相似度
sim2 = model.wv.similarity("好吃", "蟑螂")

print(f"「好吃」和「美味」的相似度：{sim1:.4f}")
print(f"「好吃」和「蟑螂」的相似度：{sim2:.4f}")

「好吃」和「美味」的相似度：0.7431
「好吃」和「蟑螂」的相似度：0.2837


In [7]:
# 任务5：执行向量运算（餐厅+聚会-安静）
result = model.wv.most_similar(
    positive=["餐厅", "聚会"],
    negative=["安静"],
    topn=1
)

word_res, sim_res = result[0]
print(f"向量运算结果：餐厅 + 聚会 - 安静 = {word_res}")
print(f"相似度得分：{sim_res:.4f}")

向量运算结果：餐厅 + 聚会 - 安静 = 美食
相似度得分：0.7543
